# Principal Components and Factor Models

Companion notebook for the [slides](https://busi521.kerryback.com/latex/PCA_Factors.pdf).  We fit 3-factor models to the 25 Fama-French size/book-to-market sorted portfolios using factor analysis and principal components, construct factor-mimicking portfolios, and compare maximum Sharpe ratios.

## Data

We use monthly excess returns on the 25 size and book-to-market sorted portfolios from the Fama-French data library.

In [ ]:
import pandas as pd
import numpy as np
import pandas_datareader as pdr
import statsmodels.api as sm

startdate = '1980-01-01'

# 25 size/BM sorted portfolio returns
Rets = pdr.DataReader('25_Portfolios_5x5', 'famafrench', start=startdate)[0]

# Fama-French factors and risk-free rate
FF = pdr.DataReader('F-F_Research_Data_Factors', 'famafrench', start=startdate)[0]

# convert to excess returns
Rets = Rets.apply(lambda x: x - FF['RF'])

## Factor Analysis

See slides: *ML Estimation*, *Inferring the Factors*, and *GLS Is Maximum Likelihood*.

ML estimation recovers the loading matrix $\hat{B}$ and diagonal residual covariance $\hat{D}$.  To infer the factor realizations at each date, observe that the model

443R_t - \mu = B f_t + e_t, \qquad 	ext{Cov}(e_t) = D443

is a **cross-sectional regression**: \$ observations (one per asset), \$ unknowns (the factor values \$), and \$ is the design matrix.  GLS (using the known heteroskedastic error covariance \$) gives

443\hat{f}_t = (B'D^{-1}B)^{-1}B'D^{-1}(R_t - \hat{\mu})443

The \ class in sklearn performs ML estimation via \.  We then compute the GLS factor scores from $\hat{B}$ and $\hat{D}$.

In [ ]:
from sklearn.decomposition import FactorAnalysis

fa = FactorAnalysis(n_components=3).fit(Rets)

# ML estimates
B = fa.components_.T                     # N x K loading matrix
D_inv = np.diag(1 / fa.noise_variance_)  # N x N

# GLS (Bartlett) factor scores
W = np.linalg.solve(B.T @ D_inv @ B, B.T @ D_inv)  # K x N
Factors_FA = pd.DataFrame(
    (Rets.values - fa.mean_) @ W.T,
    index=Rets.index,
    columns=range(3),
)

print("Factor means (should be ~0):")
print(Factors_FA.mean())
print("
Factor covariance:")
print(Factors_FA.cov())

### Recovering the loading matrix from time-series regressions

The factor model says  - \mu = Bf_t + e_t$.  If we run $ separate **time-series** regressions — regressing each asset's excess returns on the GLS factors — the stacked slope coefficients should recover $\hat{B}$.  The sample covariance of the residuals should be approximately diagonal, with diagonal entries close to $\hat{D}$.

In [ ]:
# run N time-series regressions: regress each asset on the 3 GLS factors
X = sm.add_constant(Factors_FA)
B_reg = pd.DataFrame(dtype=float, index=Rets.columns, columns=range(3))
resids = pd.DataFrame(dtype=float, index=Rets.index, columns=Rets.columns)
for col in Rets.columns:
    result = sm.OLS(Rets[col], X).fit()
    B_reg.loc[col] = result.params[1:]
    resids[col] = result.resid

# loading matrix from ML estimation
B_hat = pd.DataFrame(fa.components_.T, index=Rets.columns, columns=range(3))

print("Max absolute difference between regression slopes and B_hat:")
print(f"{(B_reg - B_hat).abs().values.max():.6f}")

# residual covariance: diagonal should match D_hat, off-diagonals should be small
cov_resid = resids.cov()
D_hat = pd.Series(fa.noise_variance_, index=Rets.columns)

print("
Diagonal of residual covariance vs D_hat (first 5 assets):")
diag_compare = pd.DataFrame({"Resid var": np.diag(cov_resid.values), "D_hat": D_hat.values}, index=Rets.columns)
print(diag_compare.head().round(2))

print(f"
Max |off-diagonal| element of residual covariance: ", end="")
mask = ~np.eye(25, dtype=bool)
print(f"{np.abs(cov_resid.values[mask]).max():.2f}")
print(f"Mean |off-diagonal|: {np.abs(cov_resid.values[mask]).mean():.2f}")
print(f"Mean diagonal: {np.diag(cov_resid.values).mean():.2f}")

### Why GLS creates traded factors

See slides: *GLS Is Maximum Likelihood* and *GLS Creates Traded Factors*.

Because the GLS formula \ = (\hat{B}'\hat{D}^{-1}\hat{B})^{-1}\hat{B}'\hat{D}^{-1}(R_t - \hat{\mu})$ is a linear combination of returns, each factor is exactly spanned by the returns.  We can verify this: regressing any factor on the returns gives \^2 = 1$.

In [ ]:
# regress factor 0 on the 25 returns: R-squared = 1
result = sm.OLS(Factors_FA[0], sm.add_constant(Rets)).fit()
print(f'R-squared: {result.rsquared:.6f}')

### Factor-Mimicking Portfolios (Factor Analysis)

See slides: *Constructing FMPs: Factor Analysis* and *GLS Creates Traded Factors*.

The GLS formula makes each factor a linear combination of returns â€” a portfolio return (up to a constant).  To find the portfolio weights, we run a **separate regression**: regress each *factor* on the *returns* (the reverse direction from GLS, which regresses returns on loadings).  The $R^2$ is 1 because the factors are exact linear combinations of returns.  Normalizing the coefficients to sum to one gives the FMP weights.

In [ ]:
FMP_FA = pd.DataFrame(dtype=float, index=Factors_FA.index, columns=Factors_FA.columns)
X = sm.add_constant(Rets)
for i in range(3):
    wts = sm.OLS(Factors_FA[i], X).fit().params[1:]
    wts = wts / wts.sum()
    FMP_FA[i] = Rets @ wts

FMP_FA.head()

## Principal Components

See slides: *Eigenvectors of the Covariance Matrix* and *PCA as a Factor Model*.

PCA uses the eigenvectors of $\Sigma = \text{Cov}(R_t)$.  Let $C_1$ contain the $K$ eigenvectors with the largest eigenvalues.  The factors are

$$f_t = (R_t - \hat{\mu})\,C_1$$

PCA does not force residuals to be uncorrelated.  Instead, it makes residuals small by dropping eigenvectors with small eigenvalues.

In [ ]:
from sklearn.decomposition import PCA

Factors_PCA = PCA(n_components=3).fit_transform(Rets)
Factors_PCA = pd.DataFrame(Factors_PCA, index=Rets.index, columns=range(3))
Factors_PCA.head()

We can equivalently compute the eigenvectors directly.  Note that `linalg.eigh` sorts from smallest to largest eigenvalue, so we reverse.

In [ ]:
E, C = np.linalg.eigh(Rets.cov())
C = pd.DataFrame(C, index=Rets.columns, columns=range(25))
C1 = C[[24, 23, 22]]
C1.columns = [0, 1, 2]

### Variance Explained

See slide: *How Much Variance Is Explained?*

Each eigenvalue equals the variance of the corresponding principal component.  The fraction of total variance explained by each component:

In [ ]:
fractions = E / E.sum()
print('Fraction of variance explained by each component (largest to smallest):')
print(fractions[::-1][:5])
print(f'\nFirst 3 components explain {fractions[-3:].sum():.1%} of total variance')

### Factor-Mimicking Portfolios (PCA)

See slide: *Constructing FMPs: PCA*.

The eigenvector entries are the portfolio weights.  Normalizing so the weights sum to one:

In [ ]:
FMP_PCA = Rets @ (C1 / C1.sum())
FMP_PCA.head()

## Comparing Maximum Sharpe Ratios

See slide: *Maximum Sharpe Ratios*.

The maximum Sharpe ratio from $K$ excess-return factors with mean $\mu_f$ and covariance $\Sigma_f$ is

$$\text{SR}^* = \sqrt{\mu_f'\Sigma_f^{-1}\mu_f}$$

In [ ]:
def max_sharpe(df):
    mu = df.mean()
    Sigma = df.cov()
    return np.sqrt(mu.T @ np.linalg.inv(Sigma) @ mu)

print(f'Fama-French SR:  {max_sharpe(FF[["Mkt-RF", "SMB", "HML"]]):.4f}')
print(f'FA mimicking SR: {max_sharpe(FMP_FA):.4f}')
print(f'PCA mimicking SR: {max_sharpe(FMP_PCA):.4f}')